In [4]:
%pip install tinygrad
%pip install sympy

import tinygrad as tg
import numpy as np
from typing import Tuple, Literal
import sympy as sp


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from collections.abc import Callable, Iterable
from collections import OrderedDict
from functools import cached_property


def find(iterator:Iterable, cond:Callable):
        for it in iter(iterator):
            if cond(it):
                yield element

def recursively_create_zeros(x):
        """
        Recursively loop through a tensor and convert its elements into zeros. 
        We use this when calculating the gradient vector between states of our tensor inside of computational graph.  
        """
        if hasattr(x, "shape"):
            shape = tuple(x.shape)
        elif isinstance(x, (tuple, list)):
            shape = tuple(x)
        elif isinstance(x, int):
            shape = (x,)
        else:
            raise TypeError(f"expected array or shape, got {type(x)}")
            
        # Our terminal / base case.
        if len(shape) == 0:
            return 0.0

        if len(shape) == 1:
            return [0.0 for _ in range(shape[0])]
        return [recursively_create_zeros(shape[1:]) for _ in range(shape[0])]


class CompGradGraph:
    """
    Directed acyclic graph consisting of function objects
    leaves are input tensors, roots are output tensors. 
    Chase graph from roots to leaves to automatically compute gradients using chain rule.

    Last tensor in our graph is the loss! Which we start from during back propegation to track our partial derivatives which are used separately in gradient descent.
    Gradient descent takes the negative of the gradient w.r.t the model's parameters so we can find the steepest direction of descent (normal gradients point towards direction of ascent)
    to minimize loss.
    """
    def __init__(self):
        self.nodes = {}
    

    def get_graph(self):
        g = []
        for node_id, node in sorted(self.nodes.items()):
            g.append((node_id, node))
        return g
        

    def add_node(self, node):
        if self.nodes is None:
            self.nodes = OrderedDict((node.id, node))
            
        self.nodes[node.id] = node
        print(self.nodes[node.id] == node, 'adding node')


    def add_directed_edges(self, edges:list):
        for edge_1, operation, edge_2 in edges:
            if edge_1 not in self.nodes: self.nodes[edge_1] = [operation]
            if edge_2 not in self.nodes: self.nodes[edge_2] = [operation]
            self.nodes[edge_1].append(edge_2)

__graph__ = CompGradGraph()

class Ops:
    ADD = 'add'
    MUL = 'mul' # element wise multiplication between vectors in tensor.
    SUB = 'sub' # element wise subtraction between vectors in tensor.
    MATMUL = 'matmul' # matrix multipication between tensors/
    DIV = 'div'

class Tensor:
    _next_id = 0

    def __init__(
        self,
        data,
        dtype:str = 'float32',
        req_grad:bool = False,
        grad:int | None = None,
        is_leaf:bool,
    ):
        Tensor._next_id += 1
        self.id = Tensor._next_id
        self.data = np.matrix(data, dtype=dtype)
        self.dtype = dtype
        self.req_grad = req_grad
        self.is_leaf = True
        self.grad_fn = None
        self.parents = []
        self.grad = None

        __graph__.add_node(self)

    def __add__(self, tensor):
        edges = []
        edges = [self.id, Ops.ADD, tensor.id]
        __graph__.add_directed_edges(edges)


    def calc_gradient(self, op, prev):
        """
        Rehash on gradients.
        Gradients depend on a mathematical operation between two tensors. i.e.

        x = Tensor(2)
        y = x * 2

        gradient = 2

        ---

        x = Tensor(4,4)
        y = x * 2

        gradient = 2

        ---

        x = Tensor(2)
        y = x * 2 - x / 2
        f(x) = 2x - 1/2x = 3/2x

        gradient =  1.5

        ---

        x = Tensor(2, 2, 2)
        y = x + x^2 = 1 + 2x

        gradient = 5

        Remember - The gradient vector points towards the direction of steepest ascent /
        The steepest direction maximizes the directional derivative.

        ---

        torch backwards() is called on the error tensor - 
        Autograd calculates + stores the gradients for each model parameter
        in the params .grad attribute. 

        You load your gradients (or model.parameters) into an optimizer - like SGD or ADAM
        Then you call optimizer.step() to actually initiate gradient descent (back propegation).

        Using our computational graph, we can start with a loss and track backwards (by following the chain of parents), 
        to calculate the backward derivatives (depending on the operation) until we are at the root node.
        This is instrumental for backpropegation. 

        I can use this to track from a leaf node - backwards up the computational graph 
        (depending on the OP) to recursively calc grad_x and grad_y for a function.
        """
        assert self.get_shape == y.get_shape
        
        if not op:
            return
        
        match op:
            case Ops.MUL:
                # validate that there is an actual matrix on both tensors
                if self.get_shape() != y.get_shape() and y.T.shape[0] != self.data.shape[0]:
                    raise ValueError("Non differentiatble tensors of different shape / not broadcastable.")
                # differentiating a mul op for two tensors
                grad_x, grad_y = self.backward_mul_op(prev)

                return grad_x, grad_y
            case Ops.ADD:
                return
            case Ops.MATMUL:
                return
            case Ops.DIV:
                return

    def forward_mul_op(self, x):
        """
        Forward mul op uses the product rule to create a new 
        tensor.
        d (x . y)  = x . dy + y . dx
        c = ab
        deriv of c w.t.r a = b

        Remember, we are coming up with deriv of `self` w.r.t `y`. 

        A matrix is a 2d tensor but these should be compatible too. 

        Used in backward step while calculating gradients.  
        """
        if self.data.shape[0] == y.data.shape[1]:
            # If the two tensors are broadcastable.
            #TODO: Handle this case.
            return

        y = self.data * x.data

    
    def backward_mul_op(self, grad_out, y):
        """
        z = x * y -> return (∂L/∂x, ∂L/∂y)
        """
        grad_x = grad_out * y.data
        grad_y = grad_out * self.data
        return grad_x, grad_y


    @property
    def get_shape(self):
        return self.data.shape

    @cached_property
    def T(self):
        """
        Return the transpose of the current tensor.
        The transpose is the current tensor reflected across its diagonal.
        |----
        | 1 1
        | 0 0

        x.T = 
        |-----
        | 1 0
        | 1 0
        """
        rows, cols = self.data.shape
        transposed = recursively_create_zeros((cols, rows))
        for row in range(rows):
            for col in range(cols):
                transposed[row][col] = self.data[col, row] 

        return Tensor(transposed)

    def backwards(self):
        """Jacobian of self w.r.t. ref (column vectors)"""
        m, n = self.data.size, ref.data.size

        

In [66]:
t = Tensor(data=[[2, 4], [ 6, 8]])

print(__graph__.nodes)

t2 = Tensor(2)


True adding node
{1: <__main__.Tensor object at 0x110aaf750>, 2: <__main__.Tensor object at 0x110aadc10>, 3: <__main__.Tensor object at 0x110a87ed0>}
True adding node


In [70]:
print(t.data)
print(t.T.data)

[[2. 4.]
 [6. 8.]]
[[2. 6.]
 [4. 8.]]


In [27]:

t2 = Tensor(3)

__graph__.__dict__

True adding node


{'nodes': {1: <__main__.Tensor at 0x1059a5d10>}}

In [31]:
g = __graph__.get_graph()
g[0][1]

In [23]:
# Independent numeric arrays — no symbols → zero Jacobian (no SymPy call)
t1 = Tensor(np.random.randn(10, 10))
t2 = Tensor(np.random.randn(10, 10))
t3 = Tensor(np.random.randn(10, 10))
# j = t2.calc_gradient(t1)

# Assert every instance gets the same gradient graph huzzah.
assert t1.gradient_graph == t2.gradient_graph == t3.gradient_graph
